### Shift to Colab

In [8]:
!pip install -q transformers==5.0.0 datasets evaluate jiwer albumentations kaggle
!pip install -q "peft==0.13.2"        # last version before torchao dependency
!pip install -q "torchao>=0.16.0"     # upgrade torchao in case peft version pulled still needs it

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.9 MB/s eta 0:00:00


In [1]:
# ── Install dependencies ────────────────────────────────────────────────────
# !pip install -q transformers==5.0.0 datasets evaluate jiwer albumentations kaggle peft
!pip install -q transformers==5.0.0 datasets evaluate jiwer albumentations kaggle
!pip install -q "peft==0.13.2"        # last version before torchao dependency
!pip install -q "torchao>=0.16.0"     # upgrade torchao in case peft version pulled still needs it

# ── Mount Google Drive (persistent storage) ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create persistent output dir on Drive
import os
DRIVE_DIR = "/content/drive/MyDrive/SmartStock"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/trocr-smart-stock-best", exist_ok=True)
print(f"Drive mounted. Output dir: {DRIVE_DIR}")

# ── Import Kaggle dataset ────────────────────────────────────────────────────
from google.colab import files

# Upload your kaggle.json API token (one-time per session)
# Get it from: kaggle.com → Account → API → Create New Token
files.upload()  # select kaggle.json when prompted

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download smart-stock-dataset from Kaggle
!kaggle datasets download -d maazahmad69/smart-stock-dataset -p /content/
!unzip -q /content/smart-stock-dataset.zip -d /content/smart-stock-dataset/
!rm /content/smart-stock-dataset.zip

# Verify contents
import os
for item in sorted(os.listdir("/content/smart-stock-dataset")):
    print(item)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.5 MB/s eta 0:00:00
Mounted at /content/drive
Drive mounted. Output dir: /content/drive/MyDrive/SmartStock


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/maazahmad69/smart-stock-dataset
License(s): unknown
100% 10.6G/10.6G [04:35<00:00, 41.5MB/s]

smart_stock_dataset_v2
trocr-smart-stock


#### **CORD's ground_truth** field is a raw JSON string. You must parse it and construct a flat text target from the menu array

In [2]:
import json
from collections import defaultdict
from PIL import Image

def extract_cord_crops(image: Image.Image, ground_truth_str: str) -> list:
    """
    Extract line-level crops from a CORD receipt image.

    Bounding boxes are in valid_line[].words[].quad, NOT in gt_parse.menu.
    Each group_id in valid_line = one logical receipt line.
    We group words by group_id, merge their quads into one bbox, crop it,
    and use the joined word texts as the OCR target.
    Only menu.* categories are kept (skip total, tax, header lines).

    Returns list of (cropped_PIL_image, text) pairs.
    """
    try:
        data = json.loads(ground_truth_str)
    except (json.JSONDecodeError, AttributeError):
        return []

    valid_lines = data.get("valid_line", [])
    if not valid_lines:
        return []

    # Group lines by group_id
    groups = defaultdict(list)
    for line in valid_lines:
        groups[line["group_id"]].append(line)

    w, h = image.size
    crops = []

    for gid, lines in groups.items():
        # Only keep menu lines (skip total, sub_total, etc.)
        if not any(l.get("category", "").startswith("menu.") for l in lines):
            continue

        all_words, all_xs, all_ys = [], [], []
        for line in lines:
            for word in line.get("words", []):
                text = word.get("text", "").strip()
                if text:
                    all_words.append(text)
                q = word.get("quad", {})
                if q:
                    all_xs += [q.get("x1",0), q.get("x2",0), q.get("x3",0), q.get("x4",0)]
                    all_ys += [q.get("y1",0), q.get("y2",0), q.get("y3",0), q.get("y4",0)]

        if not all_words or not all_xs:
            continue

        text = " ".join(all_words)
        x1, y1 = max(0, min(all_xs)), max(0, min(all_ys))
        x2, y2 = min(w, max(all_xs)), min(h, max(all_ys))
        if x2 <= x1 or y2 <= y1:
            continue

        crops.append((image.crop((x1, y1, x2, y2)), text))

    return crops

### **SROIE** Text Reconstruction
#### SROIE has bboxes per word. Group words by Y-coordinate (within 15px = same line), merge bounding boxes per line, crop each line region

In [3]:
def extract_sroie_crops(image: Image.Image, words: list, bboxes: list) -> list:
    """
    Group SROIE words into lines by Y-coordinate proximity.
    Each line bbox is cropped and paired with its joined text.
    Returns list of (cropped_PIL_image, text) pairs.
    """
    if not words or not bboxes:
        return []

    # Sort by top-Y of each word bbox
    items = sorted(zip(words, bboxes), key=lambda x: x[1][1])

    # Group into lines: words within 15px vertically = same line
    lines = []
    current_words, current_boxes = [items[0][0]], [items[0][1]]
    for word, box in items[1:]:
        if abs(box[1] - current_boxes[-1][1]) <= 15:
            current_words.append(word)
            current_boxes.append(box)
        else:
            lines.append((current_words, current_boxes))
            current_words, current_boxes = [word], [box]
    lines.append((current_words, current_boxes))

    w, h = image.size
    crops = []
    for line_words, line_boxes in lines:
        text = " ".join(line_words).strip()
        if not text:
            continue
        x1 = max(0, min(b[0] for b in line_boxes))
        y1 = max(0, min(b[1] for b in line_boxes))
        x2 = min(w, max(b[2] for b in line_boxes))
        y2 = min(h, max(b[3] for b in line_boxes))
        if x2 <= x1 or y2 <= y1:
            continue
        crops.append((image.crop((x1, y1, x2, y2)), text))
    return crops

### Combined Dataset Builder
OOM fix: Images encoded to PNG bytes immediately — no PIL accumulation in RAM.
Line crops: Each receipt yields multiple (crop, text) pairs instead of one (full_image, all_text).
SROIE 2x weighting: SROIE train concatenated twice — gives more weight to English receipt text vs CORD's Indonesian menus.
New dataset path: Since this is a rebuilt dataset, save to a new path to avoid loading the old full-image version.

In [4]:
import io
import json
from pathlib import Path
from datasets import load_dataset, Dataset, DatasetDict
from PIL import Image

# New path — line-crop dataset is incompatible with old full-image dataset
SAVE_PATH = Path("/content/smart-stock-dataset/smart_stock_dataset_v2")
if not SAVE_PATH.exists():
    SAVE_PATH = Path("/content/smart_stock_dataset_v2")  # fallback if rebuilding fresh

def pil_to_bytes(img: Image.Image) -> bytes:
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="PNG")
    return buf.getvalue()

def iter_to_dataset(iterator) -> Dataset:
    """
    Convert an iterator of (PIL Image, text) tuples into a Dataset.
    Encodes each image to bytes immediately — never accumulates PIL objects in RAM.
    """
    img_bytes, texts = [], []
    for img, text in iterator:
        img_bytes.append(pil_to_bytes(img))
        texts.append(text)
    return Dataset.from_dict({"image_bytes": img_bytes, "text": texts})

def build_and_save_dataset():
    """
    Build line-crop CORD + SROIE dataset and save to disk.
    On subsequent runs, loads directly from disk — no reprocessing.
    """
    if SAVE_PATH.exists():
        print(f"Dataset found at {SAVE_PATH} — loading from disk...")
        return DatasetDict.load_from_disk(str(SAVE_PATH))

    print("Building line-crop dataset from scratch...")

    # --- CORD — line crops via bounding boxes ---
    print("Loading CORD...")
    cord = load_dataset("naver-clova-ix/cord-v2")

    def cord_iter(split):
        for ex in cord[split]:
            for crop, text in extract_cord_crops(ex["image"], ex["ground_truth"]):
                yield crop, text

    cord_train      = iter_to_dataset(cord_iter("train"))
    cord_validation = iter_to_dataset(cord_iter("validation"))
    cord_test       = iter_to_dataset(cord_iter("test"))
    print(f"  CORD train: {len(cord_train)} | val: {len(cord_validation)} | test: {len(cord_test)}")
    del cord

    # --- SROIE — line crops via word bbox grouping ---
    print("Loading SROIE...")
    sroie = load_dataset("sizhkhy/SROIE")

    def sroie_iter(split):
        for ex in sroie[split]:
            for crop, text in extract_sroie_crops(ex["images"], ex["words"], ex["bboxes"]):
                yield crop, text

    sroie_train = iter_to_dataset(sroie_iter("train"))
    sroie_test  = iter_to_dataset(sroie_iter("test"))
    print(f"  SROIE train: {len(sroie_train)} | test: {len(sroie_test)}")
    del sroie

    # --- Combine ---
    # SROIE train duplicated for 2x weight (more English receipt signal)
    # validation = CORD only (SROIE has no val split)
    from datasets import concatenate_datasets
    dataset_dict = DatasetDict({
        "train":      concatenate_datasets([cord_train, sroie_train, sroie_train]),
        "validation": cord_validation,
        "test":       concatenate_datasets([cord_test, sroie_test]),
    })

    # Save to Drive for persistence across sessions
    drive_save_path = Path("/content/drive/MyDrive/SmartStock/smart_stock_dataset_v2")
    drive_save_path.mkdir(parents=True, exist_ok=True)
    dataset_dict.save_to_disk(str(drive_save_path))
    # SAVE_PATH = drive_save_path  # update SAVE_PATH to Drive location

    print(f"\n✅ Saved to {SAVE_PATH}")
    print(f"   Train:      {len(dataset_dict['train'])}")
    print(f"   Validation: {len(dataset_dict['validation'])}")
    print(f"   Test:       {len(dataset_dict['test'])}")
    return dataset_dict

combined_dataset = build_and_save_dataset()

Dataset found at /content/smart-stock-dataset/smart_stock_dataset_v2 — loading from disk...


#### Expected sizes (line crops):

Train: ~24,000+ examples (CORD ~6k + SROIE ~9k × 2)
Validation: ~800 examples (CORD val crops)
Test: ~6,000 examples (CORD + SROIE test crops)

#### **Augmentation**
Apply augmentation only to training images, inline during the preprocess_trocr step. The augmentation simulates real-world receipt degradation.

In [5]:
import albumentations as A
import cv2
import numpy as np
from PIL import Image, ImageOps

receipt_augmentation = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=(-0.4, 0.15), p=0.6), # Stronger thermal fade
    A.GaussNoise(p=0.5),                                               # Scanner noise
    A.Rotate(limit=8, border_mode=cv2.BORDER_REPLICATE, p=0.5),        # Crumple tilt
    A.Perspective(scale=(0.02, 0.08), p=0.4),                          # Photo angle
    A.MotionBlur(blur_limit=5, p=0.3),                                 # Shaky photo
    A.ImageCompression(p=0.5),                                         # JPEG artifact
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),                          # Focus blur
    A.RandomShadow(p=0.2),                                             # Shadows on receipt
])

def apply_augmentation(pil_image: Image.Image) -> Image.Image:
    """Pad tiny line crops to min 32px height, then augment."""
    pil_image = pil_image.convert("RGB")
    if pil_image.height < 32:
        pad = 32 - pil_image.height
        pil_image = ImageOps.expand(pil_image, border=(0, pad//2, 0, pad - pad//2), fill=255)
    img_np = np.array(pil_image)
    augmented = receipt_augmentation(image=img_np)["image"]
    return Image.fromarray(augmented)

### Preprocessing Function

In [6]:
import io
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

def preprocess_trocr(example, augment: bool = False):
    """
    Preprocess a single (image_bytes, text) example.
    Called per-item at access time — no upfront caching.

    Returns dict with 'pixel_values' (tensor) and 'labels' (tensor).
    """
    image = Image.open(io.BytesIO(example["image_bytes"])).convert("RGB")

    if augment:
        image = apply_augmentation(image)

    pixel_values = processor(images=image, return_tensors="pt").pixel_values

    labels = processor.tokenizer(
        example["text"],
        padding="max_length",
        max_length=128,
        truncation=True,
    ).input_ids

    labels = [
        t if t != processor.tokenizer.pad_token_id else -100
        for t in labels
    ]

    return {
        "pixel_values": pixel_values.squeeze(),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


class TrOCRDataset(TorchDataset):
    """
    On-the-fly preprocessing — processes each example at access time.
    Replaces .map() which caused either:
      - OOM (keep_in_memory=True on 31k examples)
      - OSError Errno 28 (cache_file_name on full disk)
    Zero disk usage, zero RAM accumulation, full dataset used.
    Compatible with Seq2SeqTrainer — accepts any PyTorch Dataset.
    """
    def __init__(self, hf_dataset, augment: bool = False):
        self.data    = hf_dataset
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return preprocess_trocr(self.data[idx], augment=self.augment)


train_dataset = TrOCRDataset(combined_dataset["train"],      augment=True)
val_dataset   = TrOCRDataset(combined_dataset["validation"], augment=False)
test_dataset  = TrOCRDataset(combined_dataset["test"],       augment=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Train: 31057 | Val: 221 | Test: 8301


### Model Setup

In [14]:
from transformers import VisionEncoderDecoderModel, GenerationConfig
from peft import LoraConfig, get_peft_model, TaskType

model = VisionEncoderDecoderModel.from_pretrained(
    "/content/drive/MyDrive/SmartStock/trocr-smart-stock-best"
)

# ── Freeze encoder entirely ──────────────────────────────────────────────────
# ViT encoder is already well-trained on printed text — no adaptation needed.
# Freezing cuts ~half the parameters, frees significant VRAM.
for param in model.encoder.parameters():
    param.requires_grad = False

# ── Apply LoRA to decoder attention layers only ──────────────────────────────
# TrOCR decoder is RoBERTa-based. LoRA targets q_proj and v_proj in each
# decoder self-attention and cross-attention layer.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                          # rank — higher = more capacity, more params
    lora_alpha=32,                 # scaling factor (alpha/r = 2 is standard)
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # decoder attention projections
    bias="none",
    modules_to_save=None,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1-3M trainable vs 333M total (~0.5-1%)

# ── Required decoder config ──────────────────────────────────────────────────
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

model.generation_config.eos_token_id         = processor.tokenizer.sep_token_id
model.generation_config.early_stopping       = True
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.length_penalty       = 2.0
model.generation_config.num_beams            = 4

loading configuration file /content/drive/MyDrive/SmartStock/trocr-smart-stock-best/config.json
Model config VisionEncoderDecoderConfig {
  "architectures": [
    "VisionEncoderDecoderModel"
  ],
  "decoder": {
    "_name_or_path": "",
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "add_cross_attention": true,
    "architectures": null,
    "attention_dropout": 0.0,
    "bos_token_id": 0,
    "chunk_size_feed_forward": 0,
    "classifier_dropout": 0.0,
    "cross_attention_hidden_size": 768,
    "d_model": 1024,
    "decoder_attention_heads": 16,
    "decoder_ffn_dim": 4096,
    "decoder_layerdrop": 0.0,
    "decoder_layers": 12,
    "decoder_start_token_id": 2,
    "dropout": 0.1,
    "dtype": "float32",
    "eos_token_id": 2,
    "finetuning_task": null,
    "id2label": {
      "0": "LABEL_0",
      "1": "LABEL_1"
    },
    "init_std": 0.02,
    "is_decoder": true,
    "is_encoder_decoder": false,
    "label2id": {
      "LABEL_0": 0,
      "LABEL_1": 1
    },

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

loading configuration file /content/drive/MyDrive/SmartStock/trocr-smart-stock-best/generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": true,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": 2,
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 2.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 3,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 4,
  "num_return_sequences": 1,
  "output_scores": false,
  "pad_token_id": 1,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return_dict_in_generate": false,
  "target_lookbehind": 10,
  "temperature": 1.0,
  "top_k": 50,
  "top_p": 1.0,
  "typical_p": 1.0,
  "use_cache": false
}



trainable params: 1,523,712 || all params: 335,445,504 || trainable%: 0.4542


### Training Arguments

In [15]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/SmartStock/trocr-smart-stock",

    num_train_epochs=5,
    per_device_train_batch_size=8,     # restored to 8 — encoder frozen frees VRAM
    per_device_eval_batch_size=8,

    learning_rate=3e-4,                # higher LR valid for LoRA — only adapter weights move
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    save_total_limit=5,

    predict_with_generate=True,
    generation_max_length=128,

    fp16=True,
    dataloader_num_workers=2,

    logging_dir="./logs",
    logging_steps=50,
    log_level="info",
    report_to="none",
)

PyTorch: setting up devices
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Metrics

In [16]:
!pip install jiwer

In [17]:
import numpy as np
from jiwer import cer, wer

def compute_metrics(pred):
    pred_ids   = pred.predictions
    labels_ids = pred.label_ids

    # pred_ids may be float logits — argmax to get token ids
    if pred_ids.dtype != np.int64 and pred_ids.ndim == 3:
        pred_ids = np.argmax(pred_ids, axis=-1)

    # Clip to valid vocab range to prevent OverflowError during decode
    vocab_size = processor.tokenizer.vocab_size
    pred_ids   = np.clip(pred_ids, 0, vocab_size - 1)

    # Decode predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)

    # Replace -100 (padding mask) with pad token id before decoding
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    return {
        "cer": round(cer(label_str, pred_str), 4),
        "wer": round(wer(label_str, pred_str), 4),
    }

### Hyperparameter Search with **Optuna**
After the first clean training run with line crops, use Optuna (Bayesian optimization) via HuggingFace built-in hyperparameter_search to find optimal values.

In [18]:
!pip install optuna -q

import gc
import torch
import copy
from transformers import Seq2SeqTrainer
from peft import LoraConfig, get_peft_model, TaskType

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels       = torch.stack([item["labels"]       for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-4, log=True),
        "warmup_ratio":  trial.suggest_float("warmup_ratio", 0.0, 0.1),
    }

def model_init():
    gc.collect()
    torch.cuda.empty_cache()

    m = VisionEncoderDecoderModel.from_pretrained(
        "/content/drive/MyDrive/SmartStock/trocr-smart-stock-best"
    )

    # Same freeze + LoRA as Cell 14 — must match exactly
    for param in m.encoder.parameters():
        param.requires_grad = False

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none",
    )
    m = get_peft_model(m, lora_config)

    m.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    m.config.pad_token_id           = processor.tokenizer.pad_token_id
    m.config.vocab_size             = m.config.decoder.vocab_size
    m.generation_config.eos_token_id         = processor.tokenizer.sep_token_id
    m.generation_config.early_stopping       = True
    m.generation_config.no_repeat_ngram_size = 3
    m.generation_config.length_penalty       = 2.0
    m.generation_config.num_beams            = 4
    return m

search_args = copy.deepcopy(training_args)
search_args.num_train_epochs = 1
search_args.predict_with_generate = True
search_args.eval_accumulation_steps = 4
search_args.dataloader_num_workers = 0
search_args.per_device_train_batch_size = 4
search_args.per_device_eval_batch_size = 4
search_args.output_dir = "/content/optuna_search"
search_args.save_strategy = "no"
search_args.load_best_model_at_end = False
search_args.eval_strategy = "epoch"
search_args.logging_steps = 50

search_val_hf = combined_dataset["validation"].select(range(min(200, len(combined_dataset["validation"]))))
search_val = TrOCRDataset(search_val_hf, augment=False)

search_hf = combined_dataset["train"].select(range(len(combined_dataset["train"]) // 8))
search_train = TrOCRDataset(search_hf, augment=True)

search_trainer = Seq2SeqTrainer(
    model_init=model_init,
    args=search_args,
    train_dataset=search_train,
    eval_dataset=search_val,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

best_run = search_trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=4,
)

del search_trainer
gc.collect()
torch.cuda.empty_cache()

print("Best hyperparameters:", best_run.hyperparameters)
for k, v in best_run.hyperparameters.items():
    setattr(training_args, k, v)
training_args.predict_with_generate = True
print("Updated training_args:", training_args.learning_rate, training_args.warmup_ratio)

loading configuration file /content/drive/MyDrive/SmartStock/trocr-smart-stock-best/config.json
Model config VisionEncoderDecoderConfig {
  "architectures": [
    "VisionEncoderDecoderModel"
  ],
  "decoder": {
    "_name_or_path": "",
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "add_cross_attention": true,
    "architectures": null,
    "attention_dropout": 0.0,
    "bos_token_id": 0,
    "chunk_size_feed_forward": 0,
    "classifier_dropout": 0.0,
    "cross_attention_hidden_size": 768,
    "d_model": 1024,
    "decoder_attention_heads": 16,
    "decoder_ffn_dim": 4096,
    "decoder_layerdrop": 0.0,
    "decoder_layers": 12,
    "decoder_start_token_id": 2,
    "dropout": 0.1,
    "dtype": "float32",
    "eos_token_id": 2,
    "finetuning_task": null,
    "id2label": {
      "0": "LABEL_0",
      "1": "LABEL_1"
    },
    "init_std": 0.02,
    "is_decoder": true,
    "is_encoder_decoder": false,
    "label2id": {
      "LABEL_0": 0,
      "LABEL_1": 1
    },

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

loading configuration file /content/drive/MyDrive/SmartStock/trocr-smart-stock-best/generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": true,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": 2,
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 2.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 3,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 4,
  "num_return_sequences": 1,
  "output_scores": false,
  "pad_token_id": 1,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return_dict_in_generate": false,
  "target_lookbehind": 10,
  "temperature": 1.0,
  "top_k": 50,
  "top_p": 1.0,
  "typical_p": 1.0,
  "use_cache": false
}

[I 2026

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

loading configuration file /content/drive/MyDrive/SmartStock/trocr-smart-stock-best/generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": true,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": 2,
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 2.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 3,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 4,
  "num_return_sequences": 1,
  "output_scores": false,
  "pad_token_id": 1,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return_dict_in_generate": false,
  "target_lookbehind": 10,
  "temperature": 1.0,
  "top_k": 50,
  "top_p": 1.0,
  "typical_p": 1.0,
  "use_cache": false
}

***** R

...

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

loading configuration file /content/drive/MyDrive/SmartStock/trocr-smart-stock-best/generation_config.json
Generate config GenerationConfig {
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": true,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": 2,
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "length_penalty": 2.0,
  "max_length": 20,
  "min_length": 0,
  "no_repeat_ngram_size": 3,
  "num_assistant_tokens": 20,
  "num_assistant_tokens_schedule": "constant",
  "num_beam_groups": 1,
  "num_beams": 4,
  "num_return_sequences": 1,
  "output_scores": false,
  "pad_token_id": 1,
  "remove_invalid_values": false,
  "repetition_penalty": 1.0,
  "return_dict_in_generate": false,
  "target_lookbehind": 10,
  "temperature": 1.0,
  "top_k": 50,
  "top_p": 1.0,
  "typical_p": 1.0,
  "use_cache": false
}

***** R

Epoch,Training Loss,Validation Loss,Cer,Wer
1,0.904217,0.345315,0.085600,0.199800



***** Running Evaluation *****
  Num examples = 200
  Batch size = 4


Training completed. Do not forget to share your model on huggingface.co/models =)


[I 2026-06-13 21:58:11,580] Trial 3 finished with value: 0.2854 and parameters: {'learning_rate': 0.00024231897694534024, 'warmup_ratio': 0.033989959599375}. Best is trial 3 with value: 0.2854.


Best hyperparameters: {'learning_rate': 0.00024231897694534024, 'warmup_ratio': 0.033989959599375}
Updated training_args: 0.00024231897694534024 0.033989959599375


### Trainer and Training

In [ ]:
from transformers import Seq2SeqTrainer

# training_args already updated by Optuna:
# lr = 2.42e-4, warmup_ratio = 0.034
# Confirm before training:
print(f"LR: {training_args.learning_rate}")
print(f"Warmup ratio: {training_args.warmup_ratio}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

trainer = Seq2SeqTrainer(
    model=model,       # PEFT-wrapped model from Cell 14 (encoder frozen + LoRA on decoder)
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

trainer.train(resume_from_checkpoint=None)

LR: 0.00024231897694534024
Warmup ratio: 0.033989959599375
Epochs: 5
Batch size: 8


***** Running training *****
  Num examples = 31,057
  Num Epochs = 5
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 19,415
  Number of trainable parameters = 1,523,712


Epoch,Training Loss,Validation Loss


### Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.DataFrame(trainer.state.log_history)

print(history.columns)
history.head()

In [ ]:
train_logs = history[history["loss"].notna()]
eval_logs  = history[history["eval_loss"].notna()]

plt.figure(figsize=(12,6))

plt.plot(
    train_logs["step"],
    train_logs["loss"],
    label="Training Loss"
)

plt.plot(
    eval_logs["step"],
    eval_logs["eval_loss"],
    label="Validation Loss"
)

plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("TrOCR Training vs Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))

axes[0].plot(eval_logs["epoch"], eval_logs["eval_cer"], marker="o")
axes[0].set_title("CER")
axes[0].set_xlabel("Epoch")

axes[1].plot(eval_logs["epoch"], eval_logs["eval_wer"], marker="o")
axes[1].set_title("WER")
axes[1].set_xlabel("Epoch")

plt.tight_layout()
plt.show()

In [ ]:
summary = eval_logs[
    ["epoch", "eval_loss", "eval_cer", "eval_wer"]
]

summary

### Save & Export

In [ ]:
from pathlib import Path
from peft import PeftModel

DRIVE_SAVE = "/content/drive/MyDrive/SmartStock/trocr-smart-stock-best"
LOCAL_SAVE = "/content/trocr-smart-stock-best"

# ── Merge LoRA weights into base model before saving ─────────────────────────
# get_peft_model wraps the model — merged model is a plain VisionEncoderDecoderModel
# that can be loaded with from_pretrained() without peft dependency downstream.
merged_model = model.merge_and_unload()

for save_path in [DRIVE_SAVE, LOCAL_SAVE]:
    merged_model.save_pretrained(save_path)
    processor.save_pretrained(save_path)
    merged_model.generation_config.save_pretrained(save_path)
    print(f"Saved to: {save_path}")

# Verify Drive copy
expected_files = [
    "model.safetensors", "config.json", "generation_config.json",
    "tokenizer_config.json", "tokenizer.json", "processor_config.json",
]
print(f"\nDrive save verification ({DRIVE_SAVE}):")
for fname in expected_files:
    fpath = Path(DRIVE_SAVE) / fname
    exists = fpath.exists()
    size = fpath.stat().st_size / 1e6 if exists else 0
    print(f"  {'✅' if exists else '❌'} {fname} ({size:.1f} MB)")

### Evaluate on Test Set

In [ ]:
results = trainer.evaluate(test_dataset)
print(f"Test CER: {results['eval_cer']:.4f}")
print(f"Test WER: {results['eval_wer']:.4f}")

# Target benchmarks:
# CER ≤ 0.05 (5%)
# WER ≤ 0.10 (10%)

### Qualitative Evaluation

In [ ]:
import io
from PIL import Image

sample = combined_dataset["test"][0]

image = Image.open(io.BytesIO(sample["image_bytes"])).convert("RGB")

pixel_values = processor(
    image,
    return_tensors="pt"
).pixel_values.to(model.device)

generated_ids = model.generate(pixel_values)

prediction = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print("GROUND TRUTH:\n")
print(sample["text"])

print("\n")
print("="*80)

print("\nPREDICTION:\n")
print(prediction)

image